# Task 1: Exploratory Data Analysis and Data Preprocessing

## Objective
Understand the structure, content, and quality of the complaint data and prepare it for the RAG pipeline.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Define paths
data_dir = Path('../data/raw')
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Load the CFPB complaint dataset
# Note: Update the filename based on your actual data file
data_files = list(data_dir.glob('*.csv'))
if not data_files:
    # Try alternative locations
    data_files = list(Path('../data').glob('*.csv'))

if data_files:
    print(f"Found data file: {data_files[0]}")
    df = pd.read_csv(data_files[0], low_memory=False)
    print(f"Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")
else:
    print("No CSV file found. Please ensure the CFPB dataset is in data/raw/ or data/")
    print("Creating sample structure for demonstration...")
    df = pd.DataFrame()

No CSV file found. Please ensure the CFPB dataset is in data/raw/ or data/
Creating sample structure for demonstration...


## 2. Load the Dataset

In [ ]:
# Define paths
data_dir = Path('../data/raw')
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Load the CFPB complaint dataset
# Check multiple possible locations for the data file
data_files = list(data_dir.glob('*.csv'))
if not data_files:
    # Try alternative locations
    data_files = list(Path('../data').glob('*.csv'))
if not data_files:
    # Try data/data folder
    data_files = list(Path('../data/data').glob('*.csv'))

if data_files:
    import time
    file_path = data_files[0]
    file_size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"Found data file: {file_path}")
    print(f"File size: {file_size_mb:.2f} MB")
    print("Loading dataset (optimized loading - this should be faster)...")
    
    start_time = time.time()
    
    # Optimized loading: specify dtypes to speed up parsing
    # This avoids pandas having to infer types for each column
    dtype_dict = {
        'Complaint ID': 'str',
        'Product': 'str',
        'Sub-product': 'str',
        'Issue': 'str',
        'Sub-issue': 'str',
        'Company': 'str',
        'State': 'str',
        'ZIP code': 'str',
        'Consumer complaint narrative': 'str',
        'Company response to consumer': 'str',
        'Company public response': 'str',
        'Tags': 'str',
        'Consumer consent provided?': 'str',
        'Submitted via': 'str',
        'Date received': 'str',
        'Date sent to company': 'str',
        'Timely response?': 'str',
        'Consumer disputed?': 'str'
    }
    
    # Load with optimizations
    try:
        df = pd.read_csv(
            file_path,
            dtype=dtype_dict,  # Specify dtypes to avoid inference (major speedup!)
            engine='c',  # Use C engine (faster than Python engine)
            low_memory=False,  # Read entire file into memory at once
            encoding='utf-8',  # Explicit encoding
            na_values=['', 'N/A', 'n/a', 'NULL', 'null', 'None']  # Common NA values
        )
    except (KeyError, ValueError) as e:
        # If dtype dict has columns that don't exist, load without it
        print(f"Note: Some columns in dtype dict not found. Loading without dtype specification...")
        df = pd.read_csv(
            file_path,
            engine='c',
            low_memory=False,
            encoding='utf-8',
            na_values=['', 'N/A', 'n/a', 'NULL', 'null', 'None']
        )
    
    elapsed_time = time.time() - start_time
    print(f"✓ Dataset loaded in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    print(f"Dataset shape: {len(df):,} rows, {len(df.columns)} columns")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
else:
    print("No CSV file found. Please ensure the CFPB dataset is in data/raw/, data/, or data/data/")
    print("Creating sample structure for demonstration...")
    df = pd.DataFrame()

Found data file: ..\data\data\complaints.csv


## 3. Initial Data Exploration

In [ ]:
if not df.empty:
    # Display basic information
    print("Dataset Shape:", df.shape)
    print("\nColumn Names:")
    print(df.columns.tolist())
    print("\nFirst few rows:")
    display(df.head())
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())

## 4. Identify Key Columns

CFPB dataset typically has columns like:
- Product (or Product category)
- Consumer complaint narrative
- Issue, Sub-issue
- Company, State, Date received
- Complaint ID

In [ ]:
if not df.empty:
    # Standardize column names (CFPB dataset may have variations)
    column_mapping = {
        'Product': 'product',
        'Product category': 'product',
        'Consumer complaint narrative': 'narrative',
        'Complaint narrative': 'narrative',
        'Issue': 'issue',
        'Sub-issue': 'sub_issue',
        'Company': 'company',
        'State': 'state',
        'Date received': 'date_received',
        'Date Received': 'date_received',
        'Complaint ID': 'complaint_id',
        'Complaint ID': 'complaint_id'
    }
    
    # Rename columns to lowercase with underscores
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    
    # Apply additional mapping if needed
    for old_name, new_name in column_mapping.items():
        if old_name.lower().replace(' ', '_') in df.columns and new_name not in df.columns:
            df.rename(columns={old_name.lower().replace(' ', '_'): new_name}, inplace=True)
    
    print("Standardized column names:")
    print([col for col in df.columns if any(key in col for key in ['product', 'narrative', 'issue', 'company', 'state', 'date', 'complaint'])])
    
    # Identify the narrative column
    narrative_cols = [col for col in df.columns if 'narrative' in col or 'complaint' in col and 'id' not in col]
    product_cols = [col for col in df.columns if 'product' in col]
    
    print(f"\nNarrative column candidates: {narrative_cols}")
    print(f"Product column candidates: {product_cols}")

## 5. Analyze Product Distribution

In [ ]:
if not df.empty and 'product' in df.columns:
    print("Product Distribution:")
    product_dist = df['product'].value_counts()
    print(product_dist)
    
    # Visualize product distribution
    plt.figure(figsize=(14, 6))
    product_dist.plot(kind='bar')
    plt.title('Distribution of Complaints by Product', fontsize=16, fontweight='bold')
    plt.xlabel('Product', fontsize=12)
    plt.ylabel('Number of Complaints', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(output_dir / 'product_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Percentage distribution
    print("\nPercentage Distribution:")
    print((product_dist / len(df) * 100).round(2))

## 6. Analyze Narrative Length

In [ ]:
if not df.empty:
    # Find narrative column
    narrative_col = None
    for col in df.columns:
        if 'narrative' in col.lower():
            narrative_col = col
            break
    
    if narrative_col:
        # Calculate word count for each narrative
        df['narrative_word_count'] = df[narrative_col].astype(str).apply(
            lambda x: len(x.split()) if pd.notna(x) and x != 'nan' else 0
        )
        df['narrative_char_count'] = df[narrative_col].astype(str).apply(
            lambda x: len(x) if pd.notna(x) and x != 'nan' else 0
        )
        
        # Statistics
        print("Narrative Length Statistics:")
        print(f"Mean word count: {df['narrative_word_count'].mean():.2f}")
        print(f"Median word count: {df['narrative_word_count'].median():.2f}")
        print(f"Min word count: {df['narrative_word_count'].min()}")
        print(f"Max word count: {df['narrative_word_count'].max()}")
        print(f"Std word count: {df['narrative_word_count'].std():.2f}")
        
        # Identify very short and very long narratives
        short_threshold = 10  # words
        long_threshold = 1000  # words
        
        short_narratives = df[df['narrative_word_count'] < short_threshold]
        long_narratives = df[df['narrative_word_count'] > long_threshold]
        
        print(f"\nNarratives with < {short_threshold} words: {len(short_narratives):,} ({len(short_narratives)/len(df)*100:.2f}%)")
        print(f"Narratives with > {long_threshold} words: {len(long_narratives):,} ({len(long_narratives)/len(df)*100:.2f}%)")
        
        # Visualize distribution
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Histogram
        axes[0].hist(df['narrative_word_count'], bins=50, edgecolor='black', alpha=0.7)
        axes[0].axvline(df['narrative_word_count'].mean(), color='red', linestyle='--', 
                        label=f'Mean: {df["narrative_word_count"].mean():.1f}')
        axes[0].axvline(df['narrative_word_count'].median(), color='green', linestyle='--', 
                        label=f'Median: {df["narrative_word_count"].median():.1f}')
        axes[0].set_xlabel('Word Count', fontsize=12)
        axes[0].set_ylabel('Frequency', fontsize=12)
        axes[0].set_title('Distribution of Narrative Word Counts', fontsize=14, fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Box plot
        axes[1].boxplot(df['narrative_word_count'], vert=True)
        axes[1].set_ylabel('Word Count', fontsize=12)
        axes[1].set_title('Box Plot of Narrative Word Counts', fontsize=14, fontweight='bold')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_dir / 'narrative_length_distribution.png', dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print("Narrative column not found!")

## 7. Identify Complaints with/without Narratives

In [ ]:
if not df.empty and narrative_col:
    # Check for empty narratives
    empty_narratives = df[narrative_col].isna() | (df[narrative_col].astype(str).str.strip() == '') | (df[narrative_col].astype(str) == 'nan')
    
    print(f"Total complaints: {len(df):,}")
    print(f"Complaints WITH narratives: {len(df[~empty_narratives]):,} ({len(df[~empty_narratives])/len(df)*100:.2f}%)")
    print(f"Complaints WITHOUT narratives: {len(df[empty_narratives]):,} ({len(df[empty_narratives])/len(df)*100:.2f}%)")
    
    # Visualize
    plt.figure(figsize=(8, 6))
    counts = [len(df[~empty_narratives]), len(df[empty_narratives])]
    labels = ['With Narrative', 'Without Narrative']
    plt.pie(counts, labels=labels, autopct='%1.1f%%', startangle=90)
    plt.title('Complaints with/without Narratives', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_dir / 'narrative_presence.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Filter Dataset for Required Products

In [ ]:
if not df.empty and 'product' in df.columns:
    # Define target products (case-insensitive matching)
    target_products = [
        'Credit card',
        'Credit Card',
        'Personal loan',
        'Personal Loan',
        'Savings account',
        'Savings Account',
        'Money transfers',
        'Money transfer',
        'Money Transfer'
    ]
    
    # Get unique products in dataset
    unique_products = df['product'].unique()
    print("Available products in dataset:")
    for prod in sorted(unique_products):
        print(f"  - {prod}")
    
    # Filter for target products (case-insensitive)
    df_filtered = df[df['product'].str.contains('|'.join(target_products), case=False, na=False)]
    
    print(f"\nOriginal dataset: {len(df):,} rows")
    print(f"Filtered dataset: {len(df_filtered):,} rows")
    print(f"Removed: {len(df) - len(df_filtered):,} rows")
    
    # Show distribution after filtering
    print("\nProduct distribution after filtering:")
    print(df_filtered['product'].value_counts())

## 9. Remove Empty Narratives

In [ ]:
if not df_filtered.empty and narrative_col:
    # Remove records with empty narratives
    df_filtered = df_filtered[~empty_narratives].copy()
    
    print(f"After removing empty narratives: {len(df_filtered):,} rows")
    print(f"Removed: {len(df[df['product'].str.contains('|'.join(target_products), case=False, na=False)]) - len(df_filtered):,} rows")

## 10. Text Cleaning and Normalization

In [ ]:
def clean_narrative(text):
    """
    Clean and normalize complaint narratives.
    """
    if pd.isna(text) or text == 'nan':
        return ''
    
    # Convert to string
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove common boilerplate phrases
    boilerplate_phrases = [
        r'\bi am writing to file a complaint[^.]*\.',
        r'\bi am writing to complain[^.]*\.',
        r'\bthis is a complaint[^.]*\.',
        r'\bi would like to file a complaint[^.]*\.',
    ]
    
    for phrase in boilerplate_phrases:
        text = re.sub(phrase, '', text, flags=re.IGNORECASE)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    # Optional: Remove special characters (keep alphanumeric, spaces, and basic punctuation)
    # Uncomment if needed:
    # text = re.sub(r'[^a-z0-9\s\.\,\!\?]', '', text)
    
    return text

if not df_filtered.empty and narrative_col:
    print("Cleaning narratives...")
    
    # Create cleaned narrative column
    df_filtered['narrative_cleaned'] = df_filtered[narrative_col].apply(clean_narrative)
    
    # Check for any narratives that became empty after cleaning
    empty_after_cleaning = df_filtered['narrative_cleaned'].str.strip() == ''
    print(f"Narratives that became empty after cleaning: {empty_after_cleaning.sum()}")
    
    # Remove those as well
    df_filtered = df_filtered[~empty_after_cleaning].copy()
    
    print(f"Final filtered dataset: {len(df_filtered):,} rows")
    
    # Show example of cleaned text
    print("\nExample of original vs cleaned narrative:")
    sample_idx = df_filtered.index[0]
    print(f"\nOriginal:\n{df_filtered.loc[sample_idx, narrative_col][:200]}...")
    print(f"\nCleaned:\n{df_filtered.loc[sample_idx, 'narrative_cleaned'][:200]}...")

## 11. Final Dataset Summary

In [ ]:
if not df_filtered.empty:
    print("Final Dataset Summary:")
    print(f"Total rows: {len(df_filtered):,}")
    print(f"Total columns: {len(df_filtered.columns)}")
    print(f"\nProduct distribution:")
    print(df_filtered['product'].value_counts())
    
    # Recalculate word counts for cleaned narratives
    df_filtered['cleaned_word_count'] = df_filtered['narrative_cleaned'].apply(
        lambda x: len(x.split()) if x else 0
    )
    
    print(f"\nCleaned narrative statistics:")
    print(f"Mean word count: {df_filtered['cleaned_word_count'].mean():.2f}")
    print(f"Median word count: {df_filtered['cleaned_word_count'].median():.2f}")
    print(f"Min word count: {df_filtered['cleaned_word_count'].min()}")
    print(f"Max word count: {df_filtered['cleaned_word_count'].max()}")

## 12. Save Filtered Dataset

In [ ]:
if not df_filtered.empty:
    # Ensure we have a complaint_id column (create if missing)
    if 'complaint_id' not in df_filtered.columns:
        df_filtered['complaint_id'] = range(1, len(df_filtered) + 1)
    
    # Select key columns for the filtered dataset
    key_columns = ['complaint_id', 'product', 'narrative_cleaned']
    
    # Add other available columns
    additional_cols = ['issue', 'sub_issue', 'company', 'state', 'date_received']
    for col in additional_cols:
        if col in df_filtered.columns:
            key_columns.append(col)
    
    # Create final dataset
    df_final = df_filtered[key_columns].copy()
    
    # Rename narrative_cleaned to narrative for consistency
    df_final.rename(columns={'narrative_cleaned': 'narrative'}, inplace=True)
    
    # Save to CSV
    output_path = output_dir / 'filtered_complaints.csv'
    df_final.to_csv(output_path, index=False)
    print(f"Filtered dataset saved to: {output_path}")
    print(f"Shape: {df_final.shape}")
    
    # Display first few rows
    display(df_final.head())